# ABSA Indodax — FASE 1: Pengumpulan dan Persiapan Data

**Versi V5.4 — Empat Fase dengan Checkpoint**

Notebook ini dijalankan satu kali untuk memuat data hasil crawling, melakukan *light preprocessing*, mengambil sampel anotasi, membuat tiga template annotator, dan menyimpan checkpoint Fase 1. Setelah template dibuat, notebook ini tidak perlu dijalankan ulang selama checkpoint masih tersedia.


In [ ]:
# =========================================================
# 0. INSTALL LIBRARY — VERSI DIKUNCI UNTUK REPRODUCIBILITY
# =========================================================
# PyTorch mengikuti versi bawaan runtime Colab agar kompatibel dengan CUDA.
# Library lain dikunci agar perubahan versi tidak mengubah perilaku pipeline.

!pip -q install \
    "transformers==4.44.2" \
    "accelerate==0.34.2" \
    "scikit-learn==1.5.2" \
    "wordcloud==1.9.3" \
    "openpyxl==3.1.5" \
    "xlsxwriter==3.2.0"


In [ ]:
# =========================================================
# 1. IMPORT LIBRARY, SEED, DAN KONFIGURASI UTAMA — V5.4
# =========================================================

import os
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import re
import json
import math
import random
import shutil
import warnings
import zlib
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import sklearn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


def set_global_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception as exc:
        print("Peringatan determinisme:", exc)


set_global_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("SEED:", SEED)

# -----------------------------
# Workspace pipeline terpisah
# -----------------------------
USE_GOOGLE_DRIVE = True
PROJECT_NAME = "ABSA_Indodax_Pipeline_V5_4_4Fase"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
    except Exception as exc:
        print("Google Drive tidak dapat dimount. Workspace lokal digunakan:", exc)
        PROJECT_DIR = Path.cwd() / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd() / PROJECT_NAME

STAGE1_DIR = PROJECT_DIR / "01_data_preparation"
STAGE2_DIR = PROJECT_DIR / "02_annotation_processing"
STAGE3_DIR = PROJECT_DIR / "03_model_training_evaluation"
for path in [PROJECT_DIR, STAGE1_DIR, STAGE2_DIR, STAGE3_DIR]:
    path.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = "indodax_reviews.xlsx"  # hanya dipakai Notebook Bagian 1
OUT_DIR = STAGE1_DIR

# Tahap 1 dipertahankan agar preprocessing tidak perlu diulang.
# Tahap 2–3 dibersihkan secara default agar hasil lama tidak tercampur.
RESET_STAGE1 = False
RESET_OUTPUT_DIR = RESET_STAGE1
if RESET_OUTPUT_DIR and OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "models"
for path in [OUT_DIR, FIG_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR tahap aktif:", OUT_DIR)

# -----------------------------
# Label penelitian
# -----------------------------
# Empat aspek berikut adalah aspek substantif yang dianalisis.
RESEARCH_ASPECT_LABELS = [
    "antarmuka_pengguna",
    "keamanan",
    "transaksi",
    "layanan_pelanggan",
]

# tidak_relevan dipakai sebagai kelas operasional model aspek untuk mencegah
# seluruh ulasan umum dipaksa masuk ke salah satu dari empat aspek penelitian.
IRRELEVANT_ASPECT_LABEL = "tidak_relevan"
ASPECT_MODEL_LABELS = RESEARCH_ASPECT_LABELS + [IRRELEVANT_ASPECT_LABEL]

# Alias kompatibilitas untuk fungsi lama: sekarang berisi lima kelas model aspek.
ASPECT_LABELS = ASPECT_MODEL_LABELS
ANNOTATION_ASPECT_LABELS = ASPECT_MODEL_LABELS
ASPECT_LABELING_POLICY = "single_dominant_aspect"

SENTIMENT_LABELS = ["negatif", "netral", "positif"]

ASPECT_DISPLAY_LABELS = {
    "antarmuka_pengguna": "Antarmuka pengguna / UI-UX",
    "keamanan": "Keamanan akun dan transaksi",
    "transaksi": "Transaksi, termasuk biaya, deposit, withdraw, pajak, order, dan jual-beli",
    "layanan_pelanggan": "Layanan pelanggan / customer service",
    "tidak_relevan": "Tidak relevan / tidak membahas empat aspek penelitian",
}

ASPECT_DEFINITIONS = {
    "antarmuka_pengguna": "Tampilan, kemudahan penggunaan, navigasi, performa aplikasi, bug, crash, loading, update, dan pengalaman penggunaan.",
    "keamanan": "Keamanan akun, login, OTP, KYC, verifikasi, autentikasi, privasi, pembobolan, dan perlindungan dana.",
    "transaksi": "Deposit, withdraw, transfer, fee/biaya, pajak, saldo, order beli/jual, pending, gagal transaksi, dan kecepatan transaksi.",
    "layanan_pelanggan": "Customer service, admin, bantuan, komplain, tiket, email, respons, dan penyelesaian masalah.",
    "tidak_relevan": "Ulasan umum atau ulasan yang tidak membahas satu pun dari empat aspek penelitian. Kelas ini menjadi relevance gate pada model aspek dan tidak ikut analisis sentimen per aspek.",
}

SENTIMENT_DEFINITIONS = {
    "negatif": "Keluhan, kekecewaan, kritik, masalah, pengalaman buruk, atau ketidakpuasan.",
    "netral": "Informasi deskriptif, pertanyaan, pernyataan tanpa emosi kuat, atau polaritas tidak jelas.",
    "positif": "Pujian, kepuasan, rekomendasi, apresiasi, atau pengalaman baik.",
}

ASPECT_ALIASES = {
    "antarmuka_pengguna": "antarmuka_pengguna",
    "antarmuka": "antarmuka_pengguna",
    "ui": "antarmuka_pengguna",
    "ux": "antarmuka_pengguna",
    "user_interface": "antarmuka_pengguna",
    "pengalaman_pengguna": "antarmuka_pengguna",
    "tampilan": "antarmuka_pengguna",
    "keamanan": "keamanan",
    "security": "keamanan",
    "aman": "keamanan",
    "verifikasi": "keamanan",
    "kyc": "keamanan",
    "otp": "keamanan",
    "login": "keamanan",
    "transaksi": "transaksi",
    "biaya_transaksi": "transaksi",
    "biaya": "transaksi",
    "fee": "transaksi",
    "pajak": "transaksi",
    "withdraw": "transaksi",
    "withdrawal": "transaksi",
    "wd": "transaksi",
    "deposit": "transaksi",
    "depo": "transaksi",
    "transfer": "transaksi",
    "saldo": "transaksi",
    "order": "transaksi",
    "jual_beli": "transaksi",
    "jual beli": "transaksi",
    "transaction": "transaksi",
    "layanan_pelanggan": "layanan_pelanggan",
    "layanan": "layanan_pelanggan",
    "customer_service": "layanan_pelanggan",
    "customer service": "layanan_pelanggan",
    "cs": "layanan_pelanggan",
    "support": "layanan_pelanggan",
    "bantuan": "layanan_pelanggan",
    "tidak_relevan": "tidak_relevan",
    "tidak relevan": "tidak_relevan",
    "umum": "tidak_relevan",
    "lainnya": "tidak_relevan",
    "di_luar_aspek": "tidak_relevan",
    "di luar aspek": "tidak_relevan",
}

SENTIMENT_ALIASES = {
    "negatif": "negatif",
    "negative": "negatif",
    "minus": "negatif",
    "buruk": "negatif",
    "jelek": "negatif",
    "netral": "netral",
    "neutral": "netral",
    "biasa": "netral",
    "informasi": "netral",
    "positif": "positif",
    "positive": "positif",
    "plus": "positif",
    "baik": "positif",
    "bagus": "positif",
}

# -----------------------------
# Sampling, split, near-duplicate, dan balancing
# -----------------------------
TOTAL_REVIEW_TARGET = 80000
LABELED_SAMPLE_SIZE = 8000

TRAIN_RATIO = 0.60
VAL_RATIO = 0.20
TEST_RATIO = 0.20
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9

USE_RATING_FOR_SAMPLING_DIVERSITY = True
ANNOTATION_TEMPLATE_INCLUDE_RATING = False

GROUP_SPLIT_N_FOLDS = 5
NEAR_DUP_SIMILARITY_THRESHOLD = 0.92
NEAR_DUP_N_NEIGHBORS = 5
NEAR_DUP_MAX_FEATURES = 30000

# Dua dataset latih dibuat terpisah:
# 1) model aspek: oversampling berdasarkan kelas aspek;
# 2) model sentimen: hanya data relevan dan oversampling berdasarkan joint aspect-sentiment.
ASPECT_ROS_TARGET_MIN = 300
ASPECT_ROS_TARGET_MAX = 600
SENTIMENT_JOINT_ROS_TARGET_MIN = 100
SENTIMENT_JOINT_ROS_TARGET_MAX = 150

MIN_CLASS_COUNT_FOR_STRATIFY = 5
LOW_SUPPORT_THRESHOLD = 30

# -----------------------------
# Model dan early stopping
# -----------------------------
MODEL_NAME = "indobenchmark/indobert-base-p1"
FREEZE_BERT = False
MODEL_SELECTION_METRIC = "val_macro_f1"
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 0.002

TUNING_CONFIGS = [
    {
        "config_name": "layer2_h128_do035_lr2e-5_len160_ep5",
        "max_len": 160,
        "batch_size": 8,
        "learning_rate": 2e-5,
        "dropout": 0.35,
        "lstm_hidden": 128,
        "lstm_layers": 2,
        "epochs": 5,
    },
    {
        "config_name": "layer3_h128_do035_lr2e-5_len160_ep5",
        "max_len": 160,
        "batch_size": 8,
        "learning_rate": 2e-5,
        "dropout": 0.35,
        "lstm_hidden": 128,
        "lstm_layers": 3,
        "epochs": 5,
    },
    {
        "config_name": "layer4_h128_do035_lr2e-5_len160_ep5",
        "max_len": 160,
        "batch_size": 8,
        "learning_rate": 2e-5,
        "dropout": 0.35,
        "lstm_hidden": 128,
        "lstm_layers": 4,
        "epochs": 5,
    },
]

DEBUG_FAST_RUN = False
DEBUG_N_ROWS = 300



library_versions = {
    "python": os.sys.version,
    "torch": torch.__version__,
    "transformers": __import__("transformers").__version__,
    "sklearn": sklearn.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

run_config = {
    "pipeline_version": "V5.4_METODOLOGI",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "seed": SEED,
    "library_versions": library_versions,
    "raw_data_path": RAW_DATA_PATH,
    "total_review_target": TOTAL_REVIEW_TARGET,
    "labeled_sample_size": LABELED_SAMPLE_SIZE,
    "research_aspect_labels": RESEARCH_ASPECT_LABELS,
    "aspect_model_labels": ASPECT_MODEL_LABELS,
    "irrelevant_aspect_label": IRRELEVANT_ASPECT_LABEL,
    "sentiment_labels": SENTIMENT_LABELS,
    "aspect_labeling_policy": ASPECT_LABELING_POLICY,
    "split_strategy": "StratifiedGroupKFold_5fold_near_duplicate_group",
    "near_duplicate_similarity_threshold": NEAR_DUP_SIMILARITY_THRESHOLD,
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "aspect_ros_target_range": [ASPECT_ROS_TARGET_MIN, ASPECT_ROS_TARGET_MAX],
    "sentiment_joint_ros_target_range": [
        SENTIMENT_JOINT_ROS_TARGET_MIN,
        SENTIMENT_JOINT_ROS_TARGET_MAX,
    ],
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
    "low_support_threshold": LOW_SUPPORT_THRESHOLD,
    "model_name": MODEL_NAME,
    "freeze_bert": FREEZE_BERT,
    "model_selection_metric": MODEL_SELECTION_METRIC,
    "tuning_configs": TUNING_CONFIGS,
}
if "ANNOTATION_PROVENANCE" in globals():
    run_config["annotation_provenance"] = ANNOTATION_PROVENANCE

with open(OUT_DIR / "00_run_config.json", "w", encoding="utf-8") as file:
    json.dump(run_config, file, indent=2, ensure_ascii=False)

with open(OUT_DIR / "00_library_versions.json", "w", encoding="utf-8") as file:
    json.dump(library_versions, file, indent=2, ensure_ascii=False)

print("Konfigurasi tersimpan:", OUT_DIR / "00_run_config.json")


# Catatan Revisi Metodologi V5.4

Perbaikan yang mulai diterapkan pada tahap ini:

- istilah domain penting seperti `login`, `fee`, `withdraw`, `deposit`, `OTP`, `KYC`, `wallet`, dan `spread` dipertahankan;
- label `tidak_relevan` tidak lagi sekadar dibuang, tetapi akan dipakai sebagai kelas operasional model aspek pada tahap berikutnya;
- empat aspek penelitian tetap tidak berubah;
- template annotator tetap menggunakan satu aspek dominan dan tiga kelas sentimen;
- provenance anotasi akan dicatat pada Notebook Bagian 2.


## 2. Fungsi bantu umum

Bagian ini berisi fungsi untuk menyimpan tabel, membuat grafik, dan menghitung Fleiss' Kappa.

In [ ]:
# =========================================================
# 2. FUNGSI BANTU UMUM
# =========================================================

def save_df(df, filename, index=False):
    path = OUT_DIR / filename
    df.to_excel(path, index=index)
    print("Saved:", path)
    return path


def save_json(obj, filename):
    path = OUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    print("Saved:", path)
    return path


def normalize_label_text(value):
    """Normalisasi teks label agar variasi spasi, huruf besar, dan tanda hubung lebih aman."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in ["", "nan", "none", "null", "-", "_"]:
        return np.nan
    text = re.sub(r"[\s\-/]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text if text else np.nan


def canonicalize_aspect_label(value):
    """Memetakan variasi label aspek ke label resmi penelitian."""
    label = normalize_label_text(value)
    if pd.isna(label):
        return np.nan
    label = ASPECT_ALIASES.get(label, label)
    return label if label in ANNOTATION_ASPECT_LABELS else np.nan


def canonicalize_sentiment_label(value):
    """Memetakan variasi label sentimen ke label resmi penelitian."""
    label = normalize_label_text(value)
    if pd.isna(label):
        return np.nan
    label = SENTIMENT_ALIASES.get(label, label)
    return label if label in SENTIMENT_LABELS else np.nan


def display_label_guide():
    """Menampilkan panduan label yang konsisten dengan proposal dan notebook."""
    aspect_guide = pd.DataFrame([
        {"aspect_label": k, "display_name": ASPECT_DISPLAY_LABELS[k], "definition": ASPECT_DEFINITIONS[k]}
        for k in ANNOTATION_ASPECT_LABELS
    ])
    sentiment_guide = pd.DataFrame([
        {"sentiment_label": k, "definition": SENTIMENT_DEFINITIONS[k]}
        for k in SENTIMENT_LABELS
    ])
    display(aspect_guide)
    display(sentiment_guide)
    return aspect_guide, sentiment_guide


def plot_count_series(series, title, xlabel, ylabel, filename, rotation=30):
    plt.figure(figsize=(9, 5))
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotation, ha="right")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200)
    plt.show()
    print("Saved figure:", path)



def plot_heatmap_table(table, title, filename, xlabel="Sentimen", ylabel="Aspek"):
    """
    Membuat heatmap dari tabel/crosstab dan menyimpannya ke folder FIG_DIR.
    Fungsi ini dipakai untuk visualisasi distribusi joint label aspek x sentimen.
    """
    data = table.copy()

    # Pastikan data numerik agar dapat diplot dengan imshow.
    data = data.apply(pd.to_numeric, errors="coerce").fillna(0)

    plt.figure(figsize=(8, 5))
    im = plt.imshow(data.values, aspect="auto")
    plt.title(title)
    plt.colorbar(im, label="Jumlah Data")

    plt.xticks(
        ticks=np.arange(len(data.columns)),
        labels=[str(c) for c in data.columns],
        rotation=30,
        ha="right",
    )
    plt.yticks(
        ticks=np.arange(len(data.index)),
        labels=[str(i) for i in data.index],
    )

    max_value = data.values.max() if data.values.size > 0 else 0
    threshold = max_value / 2 if max_value > 0 else 0

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data.iloc[i, j]
            text_value = f"{value:.0f}"
            plt.text(
                j,
                i,
                text_value,
                ha="center",
                va="center",
                color="white" if value > threshold else "black",
            )

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)
    return path


def plot_confusion_matrix(cm, labels, title, filename):
    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)

    thresh = cm.max() / 2 if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
            )

    plt.ylabel("Label Aktual")
    plt.xlabel("Label Prediksi")
    plt.tight_layout()
    path = FIG_DIR / filename
    plt.savefig(path, dpi=200)
    plt.show()
    print("Saved figure:", path)


def plot_learning_curve(history_df, title_prefix, filename_prefix):
    if history_df.empty:
        return

    metrics = [
        ("train_loss", "val_loss", "Loss"),
        ("train_acc", "val_acc", "Accuracy"),
        ("train_macro_f1", "val_macro_f1", "Macro F1"),
    ]

    for train_col, val_col, y_label in metrics:
        if train_col not in history_df.columns or val_col not in history_df.columns:
            continue
        plt.figure(figsize=(8, 5))
        plt.plot(history_df["epoch"], history_df[train_col], marker="o", label=train_col)
        plt.plot(history_df["epoch"], history_df[val_col], marker="o", label=val_col)
        plt.title(f"{title_prefix} - {y_label}")
        plt.xlabel("Epoch")
        plt.ylabel(y_label)
        plt.legend()
        plt.tight_layout()
        path = FIG_DIR / f"{filename_prefix}_{y_label.lower().replace(' ', '_')}.png"
        plt.savefig(path, dpi=200)
        plt.show()
        print("Saved figure:", path)


def fleiss_kappa(label_matrix, categories):
    """Menghitung Fleiss' Kappa untuk beberapa annotator.

    Parameters
    ----------
    label_matrix : list[list[str]]
        Setiap baris berisi label dari seluruh annotator untuk satu objek.
        Fungsi ini mengharuskan jumlah annotator sama pada setiap baris dan
        seluruh label termasuk dalam ``categories``.
    categories : list[str]
        Seluruh kategori yang mungkin pada proses anotasi.
    """
    if not label_matrix:
        return np.nan

    rows = []
    expected_raters = None
    for row_idx, labels in enumerate(label_matrix):
        if expected_raters is None:
            expected_raters = len(labels)
        if len(labels) != expected_raters:
            raise ValueError(
                f"Jumlah annotator tidak konsisten pada baris {row_idx}: "
                f"expected={expected_raters}, got={len(labels)}"
            )
        unknown = [label for label in labels if label not in categories]
        if unknown:
            raise ValueError(
                f"Label di luar kategori pada baris {row_idx}: {unknown}. "
                f"Kategori resmi: {categories}"
            )
        rows.append([labels.count(cat) for cat in categories])

    M = np.asarray(rows, dtype=float)
    if M.size == 0:
        return np.nan

    row_totals = M.sum(axis=1)
    if not np.allclose(row_totals, row_totals[0]):
        raise ValueError("Jumlah penilaian per objek tidak konsisten.")

    N = M.shape[0]
    n = float(row_totals[0])
    if n <= 1:
        return np.nan

    agreement_per_item = ((M ** 2).sum(axis=1) - n) / (n * (n - 1))
    observed_agreement = agreement_per_item.mean()
    category_proportions = M.sum(axis=0) / (N * n)
    expected_agreement = (category_proportions ** 2).sum()

    if abs(1 - expected_agreement) < 1e-12:
        return np.nan
    return (observed_agreement - expected_agreement) / (1 - expected_agreement)


def majority_vote(labels):
    c = Counter([x for x in labels if pd.notna(x)])
    if not c:
        return np.nan
    top = c.most_common()
    # Jika voting seri, tidak dipakai agar label final lebih aman.
    if len(top) > 1 and top[0][1] == top[1][1]:
        return np.nan
    return top[0][0]


In [ ]:
# =========================================================
# 2A. SANITY CHECK FUNGSI WAJIB
# =========================================================

required_functions = [
    "save_df",
    "plot_count_series",
    "plot_heatmap_table",
    "plot_confusion_matrix",
    "plot_learning_curve",
    "fleiss_kappa",
    "majority_vote",
]

missing_required_functions = [fn for fn in required_functions if fn not in globals() or not callable(globals()[fn])]
if missing_required_functions:
    raise NameError(f"Fungsi wajib belum tersedia: {missing_required_functions}")

print("Sanity check fungsi wajib: OK")


## 3. Load data dan standardisasi kolom

Bagian ini membaca file hasil scraping `indodax_reviews.xlsx`.

Dataset kamu menggunakan kolom teks bernama **`review_txt`**, sehingga notebook sudah dibuat otomatis mengenali kolom tersebut sebagai kolom ulasan.

Kolom yang akan distandarkan:

- `review_txt` → `review_raw`
- `rating` → `rating`
- `date` → `date`
- `reviewId` → `review_id_original`
- `userName` → `user_name`


In [ ]:
# =========================================================
# 3. LOAD DATA DAN STANDARDISASI KOLOM
# =========================================================

# Catatan:
# Notebook ini sengaja TIDAK memuat kode scraping karena data scraping sudah tersedia.
# Dataset kamu terbaca dengan kolom: reviewId, userName, rating, review_txt, date.
# Kolom review_txt dipakai sebagai teks ulasan utama.

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"File {path} tidak ditemukan. Upload file data dengan nama {RAW_DATA_PATH} atau ubah RAW_DATA_PATH."
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError("Format file harus .xlsx, .xls, atau .csv")

raw_df = read_table_auto(RAW_DATA_PATH)
raw_df.columns = [str(c).strip() for c in raw_df.columns]

print("Shape raw data:", raw_df.shape)
print("Kolom tersedia:", list(raw_df.columns))
display(raw_df.head())

raw_data_scope = pd.DataFrame([{
    "target_total_review_proposal": TOTAL_REVIEW_TARGET,
    "raw_rows_loaded": len(raw_df),
    "default_labeled_sample_size": LABELED_SAMPLE_SIZE,
    "note": "Jika raw_rows_loaded kurang dari 80.000, sesuaikan narasi jumlah data pada laporan atau tambahkan file hasil scraping yang lengkap."
}])
display(raw_data_scope)
save_df(raw_data_scope, "01_data_scope_check.xlsx")

if DEBUG_FAST_RUN:
    raw_df = raw_df.head(DEBUG_N_ROWS).copy()
    print("DEBUG_FAST_RUN aktif. Data dipotong menjadi:", raw_df.shape)

# Deteksi kolom utama.
# review_txt ditambahkan karena dataset scraping kamu memakai nama kolom tersebut.
possible_text_cols = [
    "review_txt",       # dataset kamu
    "review",
    "content",
    "ulasan",
    "text",
    "komentar",
    "review_text",
    "reviewText",
    "at_review",
]

possible_rating_cols = [
    "rating",           # dataset kamu
    "score",
    "stars",
    "star",
    "bintang",
]

possible_date_cols = [
    "date",             # dataset kamu
    "tanggal",
    "at",
    "created_at",
    "review_date",
    "reviewCreatedVersion",
]

possible_review_id_cols = [
    "reviewId",         # dataset kamu
    "review_id",
    "id_review",
    "id",
]

possible_user_cols = [
    "userName",         # dataset kamu
    "user_name",
    "username",
    "nama_user",
    "name",
]

lower_cols = {str(c).lower().strip(): c for c in raw_df.columns}

def pick_col(possible_cols, required=False, semantic_name="kolom"):
    """Memilih nama kolom berdasarkan daftar kandidat secara case-insensitive."""
    for c in possible_cols:
        key = str(c).lower().strip()
        if key in lower_cols:
            return lower_cols[key]
    if required:
        raise ValueError(
            f"{semantic_name} wajib tidak ditemukan. Kandidat: {possible_cols}. Kolom tersedia: {list(raw_df.columns)}"
        )
    return None

TEXT_COL_RAW = pick_col(possible_text_cols, required=True, semantic_name="Kolom teks ulasan")
RATING_COL_RAW = pick_col(possible_rating_cols, required=False, semantic_name="Kolom rating")
DATE_COL_RAW = pick_col(possible_date_cols, required=False, semantic_name="Kolom tanggal")
REVIEW_ID_COL_RAW = pick_col(possible_review_id_cols, required=False, semantic_name="Kolom review ID")
USER_COL_RAW = pick_col(possible_user_cols, required=False, semantic_name="Kolom username")

print("Kolom teks ulasan:", TEXT_COL_RAW)
print("Kolom rating:", RATING_COL_RAW)
print("Kolom tanggal:", DATE_COL_RAW)
print("Kolom review ID:", REVIEW_ID_COL_RAW)
print("Kolom username:", USER_COL_RAW)

work_df = pd.DataFrame()
work_df["id"] = np.arange(len(raw_df))
work_df["review_id_original"] = raw_df[REVIEW_ID_COL_RAW].astype(str) if REVIEW_ID_COL_RAW is not None else work_df["id"].astype(str)
work_df["user_name"] = raw_df[USER_COL_RAW].astype(str) if USER_COL_RAW is not None else np.nan
work_df["review_raw"] = raw_df[TEXT_COL_RAW].astype(str)
work_df["rating"] = raw_df[RATING_COL_RAW] if RATING_COL_RAW is not None else np.nan
work_df["date"] = raw_df[DATE_COL_RAW] if DATE_COL_RAW is not None else np.nan

# Simpan sampel data raw untuk dokumentasi.
save_df(work_df.head(100), "01_sample_raw_data.xlsx")

# Visualisasi rating awal bila kolom rating tersedia.
if RATING_COL_RAW is not None:
    rating_counts = work_df["rating"].value_counts(dropna=False).sort_index()
    display(rating_counts.to_frame("count"))
    plot_count_series(
        rating_counts,
        "Distribusi Rating Data Raw",
        "Rating",
        "Jumlah",
        "01_distribusi_rating_raw.png",
        rotation=0,
    )
else:
    print("Kolom rating tidak ditemukan. Visualisasi distribusi rating dilewati.")


## 4. Preprocessing teks

Preprocessing dibuat ringan karena IndoBERT sudah mampu memahami konteks bahasa. Teks dibersihkan dari URL, karakter berlebih, spasi ganda, dan sebagian kata informal dinormalisasi.

In [ ]:
# =========================================================
# 4. PREPROCESSING TEKS — LIGHT PREPROCESSING BERBASIS DOMAIN
# =========================================================
# Istilah domain penting seperti login, fee, withdraw, deposit, OTP, KYC,
# wallet, dan spread dipertahankan agar sinyal aspek tidak hilang.

PHRASE_NORMALIZATION = {
    "log in": "login",
    "customer servis": "customer service",
    "costumer service": "customer service",
    "customer care": "customer service",
}

SLANG_DICT = {
    "gak": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "tdk": "tidak",
    "tak": "tidak",
    "gk": "tidak",
    "bgt": "banget",
    "bgtt": "banget",
    "bangett": "banget",
    "apk": "aplikasi",
    "app": "aplikasi",
    "apps": "aplikasi",
    "cs": "customer service",
    "cust": "customer",
    "tx": "transaksi",
    "wd": "withdraw",
    "wede": "withdraw",
    "depo": "deposit",
    "tf": "transfer",
    "eror": "error",
    "errorr": "error",
    "lemottt": "lemot",
    "lelet": "lambat",
    "mantul": "mantap",
}


def normalize_slang(text):
    text = str(text)
    for source, target in PHRASE_NORMALIZATION.items():
        text = re.sub(
            rf"(?<!\w){re.escape(source)}(?!\w)",
            target,
            text,
            flags=re.IGNORECASE,
        )
    tokens = text.split()
    return " ".join(SLANG_DICT.get(token, token) for token in tokens)


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"[^a-zA-ZÀ-ÿ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = normalize_slang(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


clean_df = work_df.copy()
clean_df["review_clean"] = clean_df["review_raw"].apply(clean_text)
clean_df["n_words"] = clean_df["review_clean"].str.split().apply(len)

before = len(clean_df)
clean_df = clean_df[clean_df["review_clean"].str.len() > 0].copy()
clean_df = clean_df[clean_df["n_words"] >= 2].copy()
clean_df = clean_df.drop_duplicates(subset=["review_clean"]).copy()
after = len(clean_df)

print("Jumlah data sebelum cleaning:", before)
print("Jumlah data setelah cleaning :", after)
print("Data terhapus               :", before - after)

domain_terms = [
    "login", "fee", "withdraw", "deposit", "otp", "kyc", "wallet", "spread"
]
domain_term_audit = pd.DataFrame({
    "domain_term": domain_terms,
    "count_after_preprocessing": [
        int(clean_df["review_clean"].str.contains(
            rf"\b{re.escape(term)}\b", regex=True, na=False
        ).sum())
        for term in domain_terms
    ],
})
display(domain_term_audit)

preprocessing_summary = pd.DataFrame([{
    "step": "light_preprocessing_for_indobert",
    "case_folding": True,
    "remove_url_mention_hashtag": True,
    "remove_symbols_extra_spaces": True,
    "normalize_slang": True,
    "preserve_domain_terms": True,
    "drop_empty_short_exact_duplicate": True,
    "near_duplicate_handling": "Dilakukan saat group split pada Notebook Bagian 2.",
    "aggressive_stopword_removal": False,
    "stemming": False,
    "reason": (
        "Stopword removal dan stemming agresif tidak diterapkan karena IndoBERT "
        "membutuhkan konteks kalimat; istilah domain dipertahankan sebagai sinyal aspek."
    ),
}])
display(preprocessing_summary)

save_df(preprocessing_summary, "02_preprocessing_summary.xlsx")
save_df(domain_term_audit, "02_domain_term_preservation_audit.xlsx")
display(clean_df[["id", "review_raw", "review_clean", "rating"]].head())
save_df(clean_df, "02_clean_reviews.xlsx")


## 5. Template labeling untuk 3 annotator

Bagian ini mengambil sebagian data untuk dilabeli. Setiap annotator mengisi dua kolom utama:

- `aspect_annotator`: **satu aspek dominan** yang paling utama dibahas dalam ulasan;
- `sentiment_annotator`: sentimen terhadap aspek dominan tersebut.

Aturan aspek dominan:

1. Apabila ulasan hanya membahas satu aspek, pilih aspek tersebut.
2. Apabila ulasan membahas beberapa aspek, pilih aspek yang menjadi inti keluhan, pujian, atau pembahasan paling jelas.
3. Apabila ulasan tidak membahas antarmuka pengguna, keamanan, transaksi, maupun layanan pelanggan, pilih `tidak_relevan`.
4. Label `tidak_relevan` hanya digunakan untuk penyaringan anotasi dan dikeluarkan sebelum pembangunan model. Model akhir tetap memprediksi empat aspek penelitian.

File hasil yang dibaca notebook harus bernama:

- `05_annotator1_result.xlsx`
- `05_annotator2_result.xlsx`
- `05_annotator3_result.xlsx`

Template tidak memakai kolom `catatan_pengisian`. Panduan label disediakan dalam sheet terpisah agar annotator memahami definisi label dan menerapkan aturan yang sama.


In [ ]:
# =========================================================
# 5. MEMBUAT TEMPLATE ANNOTATOR DAN CHECKPOINT TAHAP 1
# =========================================================

ANNOTATOR_TEMPLATE_FILENAMES = [
    "05_annotator1_template.xlsx",
    "05_annotator2_template.xlsx",
    "05_annotator3_template.xlsx",
]


def create_annotation_templates(df, sample_size=8000):
    sample_size = min(sample_size, len(df))

    # Rating hanya dipakai untuk menjaga keragaman sampel dan tidak ditampilkan
    # kepada annotator, sehingga rating tidak menjadi dasar label sentimen.
    if (
        USE_RATING_FOR_SAMPLING_DIVERSITY
        and "rating" in df.columns
        and df["rating"].notna().sum() > 0
    ):
        sample_parts = []
        for _, group in df.groupby("rating", dropna=False):
            group_size = max(1, int(round(sample_size * len(group) / len(df))))
            sample_parts.append(
                group.sample(n=min(group_size, len(group)), random_state=SEED)
            )

        sample_df = pd.concat(sample_parts).drop_duplicates(subset=["id"])

        if len(sample_df) > sample_size:
            sample_df = sample_df.sample(n=sample_size, random_state=SEED)
        elif len(sample_df) < sample_size:
            remaining = df[~df["id"].isin(sample_df["id"])]
            add_n = min(sample_size - len(sample_df), len(remaining))
            if add_n > 0:
                sample_df = pd.concat(
                    [
                        sample_df,
                        remaining.sample(n=add_n, random_state=SEED),
                    ]
                )
    else:
        sample_df = df.sample(n=sample_size, random_state=SEED)

    sample_df = sample_df.sort_values("id").reset_index(drop=True)

    base_cols = ["id", "review_raw", "review_clean", "date"]
    if ANNOTATION_TEMPLATE_INCLUDE_RATING and "rating" in sample_df.columns:
        base_cols.insert(3, "rating")

    template = sample_df[base_cols].copy()
    template["aspect_annotator"] = ""
    template["sentiment_annotator"] = ""

    aspect_guide = pd.DataFrame(
        [
            {
                "aspect_label": label,
                "nama_aspek": ASPECT_DISPLAY_LABELS[label],
                "definisi": ASPECT_DEFINITIONS[label],
            }
            for label in ANNOTATION_ASPECT_LABELS
        ]
    )

    sentiment_guide = pd.DataFrame(
        [
            {
                "sentiment_label": label,
                "definisi": SENTIMENT_DEFINITIONS[label],
            }
            for label in SENTIMENT_LABELS
        ]
    )

    policy_guide = pd.DataFrame(
        [
            {
                "aturan": "Satu aspek dominan",
                "penjelasan": (
                    "Pilih satu aspek yang menjadi inti pembahasan utama dalam ulasan."
                ),
            },
            {
                "aturan": "Lebih dari satu aspek",
                "penjelasan": (
                    "Pilih aspek yang menjadi inti keluhan, pujian, atau pembahasan "
                    "yang paling jelas."
                ),
            },
            {
                "aturan": "Tidak ada aspek target",
                "penjelasan": (
                    "Pilih label tidak_relevan. Data ini akan disaring sebelum pemodelan "
                    "sesuai rancangan notebook saat ini."
                ),
            },
        ]
    )

    for index, filename in enumerate(ANNOTATOR_TEMPLATE_FILENAMES, start=1):
        out_path = STAGE1_DIR / filename
        with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
            template.to_excel(writer, index=False, sheet_name="labeling")
            aspect_guide.to_excel(writer, index=False, sheet_name="panduan_aspek")
            sentiment_guide.to_excel(writer, index=False, sheet_name="panduan_sentimen")
            policy_guide.to_excel(
                writer,
                index=False,
                sheet_name="aturan_aspek_dominan",
            )
        print(f"Template annotator {index} dibuat:", out_path)

    return template


# Simpan checkpoint yang dibutuhkan oleh notebook berikutnya.
work_df.to_excel(STAGE1_DIR / "01_standardized_reviews.xlsx", index=False)
clean_df.to_excel(STAGE1_DIR / "02_clean_reviews.xlsx", index=False)

display_label_guide()
annotation_master_df = create_annotation_templates(clean_df, LABELED_SAMPLE_SIZE)
annotation_master_df.to_excel(
    STAGE1_DIR / "05_annotation_master_sample.xlsx",
    index=False,
)

stage1_manifest = pd.DataFrame(
    [
        {
            "file": "01_standardized_reviews.xlsx",
            "description": "Data hasil standardisasi kolom sebelum preprocessing.",
        },
        {
            "file": "02_clean_reviews.xlsx",
            "description": "Seluruh data hasil preprocessing ringan.",
        },
        {
            "file": "05_annotation_master_sample.xlsx",
            "description": "Master sampel yang sama untuk ketiga annotator.",
        },
        *[
            {
                "file": filename,
                "description": f"Template kosong untuk annotator {i}.",
            }
            for i, filename in enumerate(ANNOTATOR_TEMPLATE_FILENAMES, start=1)
        ],
    ]
)
stage1_manifest.to_excel(STAGE1_DIR / "STAGE1_MANIFEST.xlsx", index=False)

# ZIP berguna jika Google Drive tidak dipakai. Download ZIP ini dan unggah
# kembali ketika menjalankan Notebook Bagian 2.
stage1_zip = shutil.make_archive(
    str(PROJECT_DIR / "stage1_checkpoint"),
    "zip",
    root_dir=STAGE1_DIR,
)
print("Checkpoint Tahap 1:", stage1_zip)
print("\nTAHAP 1 SELESAI.")
print("Berikan tiga template kepada annotator.")
print(
    "Sesudah file selesai diisi, ubah namanya menjadi "
    "05_annotator1_result.xlsx, 05_annotator2_result.xlsx, "
    "dan 05_annotator3_result.xlsx, lalu jalankan Notebook Bagian 2."
)


## Hasil yang perlu disimpan

Folder `01_data_preparation` dan file `stage1_checkpoint.zip` merupakan checkpoint Tahap 1.

Berikan tiga file berikut kepada annotator:

- `05_annotator1_template.xlsx`
- `05_annotator2_template.xlsx`
- `05_annotator3_template.xlsx`

Setelah selesai, ubah nama menjadi:

- `05_annotator1_result.xlsx`
- `05_annotator2_result.xlsx`
- `05_annotator3_result.xlsx`

Simpan ketiga file tersebut pada:

`MyDrive/ABSA_Indodax_Pipeline_V5_4_4Fase/annotator_results/`

Setelah itu langsung jalankan **Notebook Bagian 2**. Notebook Bagian 1 tidak perlu dijalankan ulang.
